# CGS410 — Data Preparation (Pre-Embedding)
**Do Case Markers Reduce Contextual Encoding of Syntactic Role in LLMs?**

Archit Rahalkar (240172) · Taran Mohta (241095)

---

**Outputs of this notebook:**
- `outputs/en_tokens.csv` — all English noun tokens with syntactic role + sentence
- `outputs/hi_tokens.csv` — all Hindi noun tokens with syntactic role + sentence
- `outputs/tr_tokens.csv` — all Turkish noun tokens with syntactic role + sentence
- `outputs/en_filtered.csv` — English subset: only lemmas appearing in **both** roles
- `outputs/hi_filtered.csv` — Hindi subset: only lemmas appearing in **both** roles
- `outputs/tr_filtered.csv` — Turkish subset: only lemmas appearing in **both** roles

mBERT embedding extraction is handled in `02_embeddings.ipynb`.


## 0 · Install

In [1]:
!pip install -q conllu requests pandas

## 1 · Config

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
OUT_DIR  = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

MIN_ROLE_COUNT = 1

SUBJECT_RELS = {"nsubj", "nsubj:pass"}
OBJECT_RELS  = {"obj", "iobj"}

NOUN_UPOS = {"NOUN", "PROPN"}

TREEBANKS = {
    "en": {
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu",
        "dev"  : "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu",
        "test" : "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu",
    },
    "hi": {
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master/hi_hdtb-ud-train.conllu",
        "dev"  : "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master/hi_hdtb-ud-dev.conllu",
        "test" : "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master/hi_hdtb-ud-test.conllu",
    },
    "tr": {
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-IMST/master/tr_imst-ud-train.conllu",
        "dev"  : "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-IMST/master/tr_imst-ud-dev.conllu",
        "test" : "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-IMST/master/tr_imst-ud-test.conllu",
    },
}

print("Config ready.")

Config ready.


## 2 · Download treebanks

In [ ]:
import requests

def download_file(url: str, dest: Path) -> Path:
    if dest.exists():
        print(f"  ✓ cached: {dest.name}")
        return dest
    print(f"  ↓ {dest.name} ...", end=" ", flush=True)
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"{len(r.content)/1024:.0f} KB")
    return dest

conllu_paths = {}

for lang, splits in TREEBANKS.items():
    conllu_paths[lang] = {}
    print(f"[{lang.upper()}]")
    for split, url in splits.items():
        fname = DATA_DIR / url.split("/")[-1]
        conllu_paths[lang][split] = download_file(url, fname)

print("\nAll files ready.")

[EN]
  ✓ cached: en_ewt-ud-train.conllu
  ✓ cached: en_ewt-ud-dev.conllu
  ✓ cached: en_ewt-ud-test.conllu
[HI]
  ✓ cached: hi_hdtb-ud-train.conllu
  ✓ cached: hi_hdtb-ud-dev.conllu
  ✓ cached: hi_hdtb-ud-test.conllu
[TR]
  ↓ tr_imst-ud-train.conllu ... 3047 KB
  ↓ tr_imst-ud-dev.conllu ... 874 KB
  ↓ tr_imst-ud-test.conllu ... 822 KB

All files ready.


## 3 · Parse CoNLL-U → extract noun tokens with syntactic role

For each token we record:
- Its surface form, lemma, UPOS, and dependency relation
- Its role label (`subject` or `object`)
- The full sentence string (needed later for mBERT tokenization)
- Its 1-based position in the sentence (needed to find the right mBERT sub-token)

This process is applied to **English, Hindi, and Turkish**.


In [ ]:
import conllu
import pandas as pd

def parse_treebank(paths: dict) -> pd.DataFrame:
    records = []

    for split, path in paths.items():
        raw = path.read_text(encoding="utf-8")
        sentences = conllu.parse(raw)

        for sent in sentences:
            int_tokens = [t for t in sent if isinstance(t["id"], int)]
            sentence_str = " ".join(t["form"] for t in int_tokens)
            sent_id = sent.metadata.get("sent_id", "")

            for token in int_tokens:
                if token["upos"] not in NOUN_UPOS:
                    continue

                deprel = token["deprel"] or ""

                if deprel in SUBJECT_RELS:
                    role = "subject"
                elif deprel in OBJECT_RELS:
                    role = "object"
                else:
                    continue

                records.append({
                    "sent_id"   : sent_id,
                    "split"     : split,
                    "token_idx" : token["id"],
                    "form"      : token["form"],
                    "lemma"     : (token["lemma"] or token["form"]).lower(),
                    "upos"      : token["upos"],
                    "deprel"    : deprel,
                    "role"      : role,
                    "sentence"  : sentence_str,
                })

    return pd.DataFrame(records)


print("Parsing English EWT ...")
en_df = parse_treebank(conllu_paths["en"])
print(f"  {len(en_df):,} tokens | {en_df['role'].value_counts().to_dict()}")

print("Parsing Hindi HDTB ...")
hi_df = parse_treebank(conllu_paths["hi"])
print(f"  {len(hi_df):,} tokens | {hi_df['role'].value_counts().to_dict()}")

print("Parsing Turkish IMST ...")
tr_df = parse_treebank(conllu_paths["tr"])
print(f"  {len(tr_df):,} tokens | {tr_df['role'].value_counts().to_dict()}")

Parsing English EWT ...
  16,930 tokens | {'object': 9317, 'subject': 7613}
Parsing Hindi HDTB ...
  29,877 tokens | {'subject': 16251, 'object': 13626}
Parsing Turkish IMST ...
  5,611 tokens | {'subject': 3105, 'object': 2506}


## 4 · Filter to lemmas appearing in both roles

Per the proposal, we only retain lemmas that appear **at least once as a subject and at least once as an object** so we can directly compare the same word's representation across syntactic roles.

In [ ]:
def filter_dual_role_lemmas(df: pd.DataFrame, min_count: int = 1):
    role_counts = (
        df.groupby(["lemma", "role"])
          .size()
          .unstack(fill_value=0)
    )
    for col in ["subject", "object"]:
        if col not in role_counts.columns:
            role_counts[col] = 0

    dual_role_lemmas = role_counts[
        (role_counts["subject"] >= min_count) &
        (role_counts["object"]  >= min_count)
    ].index

    filtered = df[df["lemma"].isin(dual_role_lemmas)].copy()
    return filtered, role_counts


en_filtered, en_role_counts = filter_dual_role_lemmas(en_df, MIN_ROLE_COUNT)
hi_filtered, hi_role_counts = filter_dual_role_lemmas(hi_df, MIN_ROLE_COUNT)
tr_filtered, tr_role_counts = filter_dual_role_lemmas(tr_df, MIN_ROLE_COUNT)

for lang, full, filt in [("English", en_df, en_filtered), ("Hindi", hi_df, hi_filtered), ("Turkish", tr_df, tr_filtered)]:
    print(f"[{lang}]")
    print(f"  Before : {len(full):,} tokens, {full['lemma'].nunique():,} lemmas")
    print(f"  After  : {len(filt):,} tokens, {filt['lemma'].nunique():,} lemmas")
    print(f"  Kept   : {len(filt)/len(full)*100:.1f}% of tokens")
    print()

[English]
  Before : 16,930 tokens, 4,232 lemmas
  After  : 12,429 tokens, 1,315 lemmas
  Kept   : 73.4% of tokens

[Hindi]
  Before : 29,877 tokens, 4,728 lemmas
  After  : 24,478 tokens, 1,695 lemmas
  Kept   : 81.9% of tokens

[Turkish]
  Before : 5,611 tokens, 1,850 lemmas
  After  : 3,638 tokens, 557 lemmas
  Kept   : 64.8% of tokens



## 4.5 · Balance roles (equal subject/object counts)

In [ ]:
def balance_roles(df: pd.DataFrame):
    subject_tokens = df[df['role'] == 'subject']
    object_tokens = df[df['role'] == 'object']
    
    min_count = min(len(subject_tokens), len(object_tokens))
    
    balanced = pd.concat([
        subject_tokens.sample(n=min_count, random_state=42),
        object_tokens.sample(n=min_count, random_state=42)
    ]).reset_index(drop=True)
    
    return balanced

en_balanced = balance_roles(en_filtered)
hi_balanced = balance_roles(hi_filtered)
tr_balanced = balance_roles(tr_filtered)

print(f"[English] Before balance: {len(en_filtered):,} tokens")
print(f"  {en_filtered['role'].value_counts().to_dict()}")
print(f"[English] After within-language balance: {len(en_balanced):,} tokens")
print(f"  {en_balanced['role'].value_counts().to_dict()}")
print()

print(f"[Hindi] Before balance: {len(hi_filtered):,} tokens")
print(f"  {hi_filtered['role'].value_counts().to_dict()}")
print(f"[Hindi] After within-language balance: {len(hi_balanced):,} tokens")
print(f"  {hi_balanced['role'].value_counts().to_dict()}")
print()

print(f"[Turkish] Before balance: {len(tr_filtered):,} tokens")
print(f"  {tr_filtered['role'].value_counts().to_dict()}")
print(f"[Turkish] After within-language balance: {len(tr_balanced):,} tokens")
print(f"  {tr_balanced['role'].value_counts().to_dict()}")
print()

min_total = min(len(en_balanced), len(hi_balanced), len(tr_balanced))
tokens_per_role = min_total // 2

en_balanced = pd.concat([
    en_balanced[en_balanced['role'] == 'subject'].sample(n=tokens_per_role, random_state=42),
    en_balanced[en_balanced['role'] == 'object'].sample(n=tokens_per_role, random_state=42)
]).reset_index(drop=True)

hi_balanced = pd.concat([
    hi_balanced[hi_balanced['role'] == 'subject'].sample(n=tokens_per_role, random_state=42),
    hi_balanced[hi_balanced['role'] == 'object'].sample(n=tokens_per_role, random_state=42)
]).reset_index(drop=True)

tr_balanced = pd.concat([
    tr_balanced[tr_balanced['role'] == 'subject'].sample(n=tokens_per_role, random_state=42),
    tr_balanced[tr_balanced['role'] == 'object'].sample(n=tokens_per_role, random_state=42)
]).reset_index(drop=True)

print("=" * 50)
print("FINAL BALANCED DATASETS (equal across languages):")
print(f"[English] {len(en_balanced):,} tokens | {en_balanced['role'].value_counts().to_dict()}")
print(f"[Hindi]   {len(hi_balanced):,} tokens | {hi_balanced['role'].value_counts().to_dict()}")
print(f"[Turkish] {len(tr_balanced):,} tokens | {tr_balanced['role'].value_counts().to_dict()}")
print("=" * 50)

[English] Before balance: 12,429 tokens
  {'object': 6917, 'subject': 5512}
[English] After within-language balance: 11,024 tokens
  {'subject': 5512, 'object': 5512}

[Hindi] Before balance: 24,478 tokens
  {'subject': 13317, 'object': 11161}
[Hindi] After within-language balance: 22,322 tokens
  {'subject': 11161, 'object': 11161}

[Turkish] Before balance: 3,638 tokens
  {'subject': 1921, 'object': 1717}
[Turkish] After within-language balance: 3,434 tokens
  {'subject': 1717, 'object': 1717}

FINAL BALANCED DATASETS (equal across languages):
[English] 3,434 tokens | {'subject': 1717, 'object': 1717}
[Hindi]   3,434 tokens | {'subject': 1717, 'object': 1717}
[Turkish] 3,434 tokens | {'subject': 1717, 'object': 1717}


## 5 · Save

In [11]:
en_df.to_csv(OUT_DIR / "en_tokens.csv",   index=False)
hi_df.to_csv(OUT_DIR / "hi_tokens.csv",   index=False)
tr_df.to_csv(OUT_DIR / "tr_tokens.csv",   index=False)
en_filtered.to_csv(OUT_DIR / "en_filtered.csv", index=False)
hi_filtered.to_csv(OUT_DIR / "hi_filtered.csv", index=False)
tr_filtered.to_csv(OUT_DIR / "tr_filtered.csv", index=False)
en_balanced.to_csv(OUT_DIR / "en_balanced.csv", index=False)
hi_balanced.to_csv(OUT_DIR / "hi_balanced.csv", index=False)
tr_balanced.to_csv(OUT_DIR / "tr_balanced.csv", index=False)

print("Saved to outputs/:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")


Saved to outputs/:
  cosine_distance_by_layer.csv  (1 KB)
  en_balanced.csv  (729 KB)
  en_embeddings.npz  (398584 KB)
  en_filtered.csv  (2618 KB)
  en_lemma_dist_layer8.csv  (28 KB)
  en_meta.csv  (886 KB)
  en_tokens.csv  (3601 KB)
  hi_balanced.csv  (1283 KB)
  hi_embeddings.npz  (398708 KB)
  hi_filtered.csv  (9117 KB)
  hi_lemma_dist_layer8.csv  (32 KB)
  hi_meta.csv  (733 KB)
  hi_tokens.csv  (11123 KB)
  probe_accuracy_by_layer.csv  (1 KB)
  tr_balanced.csv  (550 KB)
  tr_filtered.csv  (582 KB)
  tr_tokens.csv  (914 KB)
